# Pipeline dự đoán giá cà phê — chạy từng dòng

Cách dùng: chạy **lần lượt từng ô từ trên xuống** (bấm `Shift + Enter`). Mỗi ô chỉ có 1–2 dòng code, và kết quả hiện ngay bên dưới.
Bạn nên tự gõ lại từng ô thay vì copy, để nhớ lâu hơn.

Mục tiêu: biến **bảng giá cà phê + bảng thời tiết** thành **bảng dùng để dạy model**.

1. Làm sạch bảng giá cà phê
2. Làm sạch bảng thời tiết
3. Ghép hai bảng theo ngày
4. Thêm các cột "gợi ý" (feature) và cột "đáp án" (target)

In [53]:
from pathlib import Path
import pandas as pd

# 1. Tự động xác định thư mục gốc project
# Nếu đang ở trong thư mục con (vd: notebooks/), lấy thư mục cha. Nếu đang ở root, giữ nguyên.
current_dir = Path.cwd()
PROJECT_ROOT = (
    current_dir.parent
    if (current_dir.parent / "data" / "raw").exists()
    else current_dir
)

# 2. Đường dẫn file nội bộ (giống hệt pipeline.py)
COFFEE_RAW = (
    PROJECT_ROOT / "data" / "raw" / "coffee" / "coffee_daklak_3y_complete.csv"
)
WEATHER_RAW = PROJECT_ROOT / "data" / "raw" / "weather" / "weather.csv"

# 3. Cơ chế fallback thông minh: nếu có file nội bộ thì đọc máy, nếu chạy trên Colab mới đọc GitHub
if COFFEE_RAW.exists() and WEATHER_RAW.exists():
  COFFEE_SOURCE = COFFEE_RAW
  WEATHER_SOURCE = WEATHER_RAW
  print(f"Đọc dữ liệu nội bộ từ: {PROJECT_ROOT / 'data' / 'raw'}")
else:
  BASE = "https://raw.githubusercontent.com/NguyenBaTam-tristan/Coffe-price-prediction-timeseries-forcasting/main/data/raw/"
  COFFEE_SOURCE = BASE + "coffee/coffee_daklak_3y_complete.csv"
  WEATHER_SOURCE = BASE + "weather/weather.csv"
  print("Không tìm thấy data/raw cục bộ, chuyển sang đọc từ GitHub.")

Không tìm thấy data/raw cục bộ, chuyển sang đọc từ GitHub.


# Phần 1. Làm sạch bảng giá cà phê

**Bước 1.1** Đọc file giá và xem 5 dòng đầu.

In [54]:
coffee = pd.read_csv(COFFEE_URL)
coffee.head()

,date,province,product,price,unit,source,price_change,source_url
0,2024-04-18,Đắk Lắk,Cà phê,122000,VND/kg,Chợ Giá,5000.0,https://chogia.vn/gia-ca-phe-dak-lak-hom-nay/
1,2024-04-19,Đắk Lắk,Cà phê,121000,VND/kg,Chợ Giá,-1000.0,https://chogia.vn/gia-ca-phe-dak-lak-hom-nay/
2,2024-04-20,Đắk Lắk,Cà phê,123500,VND/kg,Chợ Giá,2500.0,https://chogia.vn/gia-ca-phe-dak-lak-hom-nay/
3,2024-04-21,Đắk Lắk,Cà phê,123500,VND/kg,Chợ Giá,2500.0,https://chogia.vn/gia-ca-phe-dak-lak-hom-nay/
4,2024-04-22,Đắk Lắk,Cà phê,127000,VND/kg,Chợ Giá,3500.0,https://chogia.vn/gia-ca-phe-dak-lak-hom-nay/


**Bước 1.2** Bảng có bao nhiêu dòng, bao nhiêu cột? Mỗi cột đang là kiểu dữ liệu gì? (`object` nghĩa là chữ)

In [ ]:
print(coffee.shape)
print(coffee.dtypes)

(894, 8)
date             object
province         object
product          object
price             int64
unit             object
source           object
price_change    float64
source_url       object
dtype: object


**Bước 1.3** Cột `date` đang là **chữ**, chưa phải ngày. Ta đổi sang kiểu ngày.
`errors="coerce"` nghĩa là: nếu có giá trị sai thì biến thành `NaT` (ô trống kiểu ngày) thay vì báo lỗi.

In [ ]:
coffee["date"] = pd.to_datetime(coffee["date"], errors="coerce")
print(coffee["date"].dtypes)

datetime64[ns]


**Bước 1.4** Cột `price` cũng ép về kiểu số. Giá trị sai sẽ thành `NaN` (ô trống kiểu số).

In [ ]:
coffee["price"] = pd.to_numeric(coffee["price"], errors="coerce")
print(coffee["price"].dtype)

int64


**Bước 1.5** Đếm xem mỗi cột `date`, `price` có bao nhiêu ô trống.

In [ ]:
coffee[["date", "price"]].isna().sum()

,0
date,0
price,0


**Bước 1.6** Bỏ các dòng thiếu ngày hoặc thiếu giá. So sánh số dòng trước và sau.

In [ ]:
rows_before = len(coffee)
coffee = coffee.dropna(subset=["date", "price"]).copy()
print("Trước:", rows_before, "| Sau:", len(coffee))

Trước: 894 | Sau: 894


**Bước 1.7** Có ngày nào xuất hiện hai lần không? Đếm số ngày bị trùng.

In [ ]:
coffee["date"].duplicated().sum()

np.int64(0)

**Bước 1.8** Mỗi ngày chỉ giữ dòng cuối cùng (`keep="last"`), rồi sắp xếp theo ngày tăng dần.

In [ ]:
coffee = coffee.drop_duplicates(subset="date", keep="last").sort_values("date")
print(len(coffee))
coffee.head()

894


,date,province,product,price,unit,source,price_change,source_url
0,2024-04-18,Đắk Lắk,Cà phê,122000,VND/kg,Chợ Giá,5000.0,https://chogia.vn/gia-ca-phe-dak-lak-hom-nay/
1,2024-04-19,Đắk Lắk,Cà phê,121000,VND/kg,Chợ Giá,-1000.0,https://chogia.vn/gia-ca-phe-dak-lak-hom-nay/
2,2024-04-20,Đắk Lắk,Cà phê,123500,VND/kg,Chợ Giá,2500.0,https://chogia.vn/gia-ca-phe-dak-lak-hom-nay/
3,2024-04-21,Đắk Lắk,Cà phê,123500,VND/kg,Chợ Giá,2500.0,https://chogia.vn/gia-ca-phe-dak-lak-hom-nay/
4,2024-04-22,Đắk Lắk,Cà phê,127000,VND/kg,Chợ Giá,3500.0,https://chogia.vn/gia-ca-phe-dak-lak-hom-nay/


**Bước 1.9** Chỉ giữ hai cột cần dùng: `date` và `price`. Đây là bảng giá đã sạch.

In [ ]:
coffee = coffee[["date", "price"]]
coffee.tail()

,date,price
889,2026-09-24,92800
890,2026-09-25,92500
891,2026-09-26,93600
892,2026-09-27,93600
893,2026-09-28,93600


# Phần 2. Làm sạch bảng thời tiết

**Bước 2.1** File thời tiết có **2 dòng thông tin phụ** ở đầu (tọa độ, múi giờ), chưa phải bảng dữ liệu.
Ta xem 5 dòng đầu của file, **chưa đặt tên cột** (`header=None`, nên cột được đánh số 0, 1, 2...):

- Dòng 0 và dòng 1: thông tin phụ (tọa độ, độ cao, múi giờ). Đây là phần cần bỏ qua.
- Dòng 2: hàng tiêu đề thật (`time`, `temperature_2m_max (°C)`...).
- Từ dòng 3: dữ liệu từng ngày.

Nếu không bỏ 2 dòng đầu, pandas sẽ lấy dòng 0 làm tên cột và code phía sau sẽ sai.

In [ ]:
pd.read_csv(WEATHER_URL, header=None, names=range(11), nrows=5)

,0,1,2,3,4,5,6,7,8,9,10
0,latitude,longitude,elevation,utc_offset_seconds,timezone,timezone_abbreviation,NaN,NaN,NaN,NaN,NaN
1,12.688928,108.01631,418.0,25200,Asia/Bangkok,GMT+7,NaN,NaN,NaN,NaN,NaN
2,time,temperature_2m_max (°C),temperature_2m_min (°C),precipitation_sum (mm),precipitation_hours (h),rain_sum (mm),shortwave_radiation_sum (MJ/m²),relative_humidity_2m_mean (%),soil_moisture_0_to_7cm_mean (m³/m³),soil_moisture_7_to_28cm_mean (m³/m³),temperature_2m_mean (°C)
3,2024-04-18,35.8,24.4,0.70,6.0,0.70,22.62,60,0.247,0.271,29.3
4,2024-04-19,36.4,24.5,0.50,2.0,0.50,23.75,61,0.253,0.271,29.5


**Bước 2.2** Giờ đọc lại với `skiprows=2` (bỏ 2 dòng đầu).

In [ ]:
weather = pd.read_csv(WEATHER_URL, skiprows=2)
weather.head()

,time,temperature_2m_max (°C),temperature_2m_min (°C),precipitation_sum (mm),precipitation_hours (h),rain_sum (mm),shortwave_radiation_sum (MJ/m²),relative_humidity_2m_mean (%),soil_moisture_0_to_7cm_mean (m³/m³),soil_moisture_7_to_28cm_mean (m³/m³),temperature_2m_mean (°C)
0,2024-04-18,35.8,24.4,0.7,6.0,0.7,22.62,60,0.247,0.271,29.3
1,2024-04-19,36.4,24.5,0.5,2.0,0.5,23.75,61,0.253,0.271,29.5
2,2024-04-20,35.8,24.6,0.8,4.0,0.8,22.42,62,0.266,0.271,29.2
3,2024-04-21,35.6,25.1,0.9,3.0,0.9,22.22,62,0.275,0.271,29.6
4,2024-04-22,36.2,25.2,0.0,0.0,0.0,24.14,60,0.276,0.271,30.1


**Bước 2.3** Xem danh sách tên cột gốc. Tên dài, có cả đơn vị trong ngoặc, khó gõ code.

In [ ]:
list(weather.columns)

['time',
 'temperature_2m_max (°C)',
 'temperature_2m_min (°C)',
 'precipitation_sum (mm)',
 'precipitation_hours (h)',
 'rain_sum (mm)',
 'shortwave_radiation_sum (MJ/m²)',
 'relative_humidity_2m_mean (%)',
 'soil_moisture_0_to_7cm_mean (m³/m³)',
 'soil_moisture_7_to_28cm_mean (m³/m³)',
 'temperature_2m_mean (°C)']

**Bước 2.4** Tạo bảng đổi tên: tên gốc → tên ngắn gọn. `time` đổi thành `date` để khớp với bảng giá.

In [ ]:
WEATHER_COLUMNS = {
    "time": "date",
    "temperature_2m_max (°C)": "temperature_2m_max",
    "temperature_2m_min (°C)": "temperature_2m_min",
    "temperature_2m_mean (°C)": "temperature_2m_mean",
    "precipitation_sum (mm)": "precipitation_sum",
    "rain_sum (mm)": "rain_sum",
    "precipitation_hours (h)": "precipitation_hours",
    "relative_humidity_2m_mean (%)": "relative_humidity_2m_mean",
    "soil_moisture_0_to_7cm_mean (m³/m³)": "soil_moisture_0_to_7cm",
    "soil_moisture_7_to_28cm_mean (m³/m³)": "soil_moisture_7_to_28cm",
    "shortwave_radiation_sum (MJ/m²)": "shortwave_radiation_sum",
}

**Bước 2.5** Kiểm tra file có đủ cột cần thiết không. Danh sách rỗng `set()` nghĩa là đủ.

In [ ]:
missing_columns = set(WEATHER_COLUMNS) - set(weather.columns)
missing_columns

set()

**Bước 2.6** Đổi tên cột.

In [ ]:
weather = weather.rename(columns=WEATHER_COLUMNS)
weather.head()

,date,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_hours,rain_sum,shortwave_radiation_sum,relative_humidity_2m_mean,soil_moisture_0_to_7cm,soil_moisture_7_to_28cm,temperature_2m_mean
0,2024-04-18,35.8,24.4,0.7,6.0,0.7,22.62,60,0.247,0.271,29.3
1,2024-04-19,36.4,24.5,0.5,2.0,0.5,23.75,61,0.253,0.271,29.5
2,2024-04-20,35.8,24.6,0.8,4.0,0.8,22.42,62,0.266,0.271,29.2
3,2024-04-21,35.6,25.1,0.9,3.0,0.9,22.22,62,0.275,0.271,29.6
4,2024-04-22,36.2,25.2,0.0,0.0,0.0,24.14,60,0.276,0.271,30.1


**Bước 2.7** Đổi cột `date` sang kiểu ngày (giống bước 1.3).

In [ ]:
weather["date"] = pd.to_datetime(weather["date"], errors="coerce")
print(weather["date"].dtype)

datetime64[ns]


**Bước 2.8** Lấy danh sách các cột số (tất cả các cột trừ `date`) rồi ép về kiểu số.

In [ ]:
numeric_columns = [column for column in WEATHER_COLUMNS.values() if column != "date"]
numeric_columns

for column in numeric_columns:
    print(column, "->", weather[column].dtype)

temperature_2m_max -> float64
temperature_2m_min -> float64
temperature_2m_mean -> float64
precipitation_sum -> float64
rain_sum -> float64
precipitation_hours -> float64
relative_humidity_2m_mean -> int64
soil_moisture_0_to_7cm -> float64
soil_moisture_7_to_28cm -> float64
shortwave_radiation_sum -> float64


In [ ]:
weather[numeric_columns] = weather[numeric_columns].apply(pd.to_numeric, errors="coerce")
weather.dtypes

,0
date,datetime64[ns]
temperature_2m_max,float64
temperature_2m_min,float64
precipitation_sum,float64
precipitation_hours,float64
rain_sum,float64
shortwave_radiation_sum,float64
relative_humidity_2m_mean,int64
soil_moisture_0_to_7cm,float64
soil_moisture_7_to_28cm,float64


**Bước 2.9** Bỏ dòng không có ngày, bỏ ngày trùng (giữ dòng cuối), sắp xếp theo ngày.

In [ ]:
print("Số dòng ban đầu         :", len(weather))
print("Số dòng thiếu ngày (NaT):", weather["date"].isna().sum())
print("Số dòng bị trùng ngày   :", weather["date"].duplicated().sum())

weather = weather.dropna(subset=["date"])
weather = weather.drop_duplicates(subset="date", keep="last").sort_values("date")
print("Số dòng sau khi làm sạch:", len(weather))

Số dòng ban đầu         : 894
Số dòng thiếu ngày (NaT): 0
Số dòng bị trùng ngày   : 0
Số dòng sau khi làm sạch: 894


**Bước 2.10** Giữ lại cột `date` và các cột số. Đây là bảng thời tiết đã sạch.

In [ ]:
weather = weather[["date", *numeric_columns]]
weather.head()

,date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,precipitation_hours,relative_humidity_2m_mean,soil_moisture_0_to_7cm,soil_moisture_7_to_28cm,shortwave_radiation_sum
0,2024-04-18,35.8,24.4,29.3,0.7,0.7,6.0,60,0.247,0.271,22.62
1,2024-04-19,36.4,24.5,29.5,0.5,0.5,2.0,61,0.253,0.271,23.75
2,2024-04-20,35.8,24.6,29.2,0.8,0.8,4.0,62,0.266,0.271,22.42
3,2024-04-21,35.6,25.1,29.6,0.9,0.9,3.0,62,0.275,0.271,22.22
4,2024-04-22,36.2,25.2,30.1,0.0,0.0,0.0,60,0.276,0.271,24.14


# Phần 3. Ghép giá và thời tiết theo ngày

**Bước 3.1** Ghép hai bảng theo cột `date`. `how="inner"` nghĩa là chỉ giữ những ngày **có mặt ở cả hai bảng**.
So sánh số dòng của ba bảng.

In [ ]:
merged = coffee.merge(weather, on="date", how="inner")
print("coffee :", len(coffee))
print("weather:", len(weather))
print("merged :", len(merged))
merged.head()

coffee : 894
weather: 894
merged : 894


,date,price,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,precipitation_hours,relative_humidity_2m_mean,soil_moisture_0_to_7cm,soil_moisture_7_to_28cm,shortwave_radiation_sum
0,2024-04-18,122000,35.8,24.4,29.3,0.7,0.7,6.0,60,0.247,0.271,22.62
1,2024-04-19,121000,36.4,24.5,29.5,0.5,0.5,2.0,61,0.253,0.271,23.75
2,2024-04-20,123500,35.8,24.6,29.2,0.8,0.8,4.0,62,0.266,0.271,22.42
3,2024-04-21,123500,35.6,25.1,29.6,0.9,0.9,3.0,62,0.275,0.271,22.22
4,2024-04-22,127000,36.2,25.2,30.1,0.0,0.0,0.0,60,0.276,0.271,24.14


**Bước 3.2** Đổi tên cột `price` thành `coffee_price` cho rõ nghĩa, rồi sắp xếp theo ngày.

In [ ]:
merged = merged.rename(columns={"price": "coffee_price"})
merged = merged.sort_values("date")
merged.head()

,date,coffee_price,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,precipitation_hours,relative_humidity_2m_mean,soil_moisture_0_to_7cm,soil_moisture_7_to_28cm,shortwave_radiation_sum
0,2024-04-18,122000,35.8,24.4,29.3,0.7,0.7,6.0,60,0.247,0.271,22.62
1,2024-04-19,121000,36.4,24.5,29.5,0.5,0.5,2.0,61,0.253,0.271,23.75
2,2024-04-20,123500,35.8,24.6,29.2,0.8,0.8,4.0,62,0.266,0.271,22.42
3,2024-04-21,123500,35.6,25.1,29.6,0.9,0.9,3.0,62,0.275,0.271,22.22
4,2024-04-22,127000,36.2,25.2,30.1,0.0,0.0,0.0,60,0.276,0.271,24.14


# Phần 4. Tạo feature và target

**Bước 4.1** Tạo bảng `features` bằng cách **sao chép** bảng `merged`.
Bảng này **không rỗng**: nó đã có sẵn cột `date`, `coffee_price` và các cột thời tiết. Từ giờ ta chỉ **thêm cột mới** vào bên phải.

Dòng thứ hai tách riêng cột giá ra đặt tên `price` cho dễ viết.

In [ ]:
features = merged.copy()
price = features["coffee_price"]
features.head()

,date,coffee_price,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,precipitation_hours,relative_humidity_2m_mean,soil_moisture_0_to_7cm,soil_moisture_7_to_28cm,shortwave_radiation_sum
0,2024-04-18,122000,35.8,24.4,29.3,0.7,0.7,6.0,60,0.247,0.271,22.62
1,2024-04-19,121000,36.4,24.5,29.5,0.5,0.5,2.0,61,0.253,0.271,23.75
2,2024-04-20,123500,35.8,24.6,29.2,0.8,0.8,4.0,62,0.266,0.271,22.42
3,2024-04-21,123500,35.6,25.1,29.6,0.9,0.9,3.0,62,0.275,0.271,22.22
4,2024-04-22,127000,36.2,25.2,30.1,0.0,0.0,0.0,60,0.276,0.271,24.14


In [ ]:
price.head()

,coffee_price
0,122000
1,121000
2,123500
3,123500
4,127000


### Feature loại 1: giá của những ngày trước (lag)

`price.shift(1)` dịch cột giá **xuống 1 dòng**, nên mỗi dòng sẽ thấy giá của **ngày hôm trước**.
Dòng đầu tiên chưa có ngày trước nên là `NaN`.

Để không làm rối bảng chính, ta thử trên một **bảng tạm** tên `demo` (chỉ có ngày và giá). Bảng `features` chưa bị thay đổi.

In [ ]:
demo = features[["date", "coffee_price"]].copy()
demo["price_lag_1"] = price.shift(1)
demo.head(5)

,date,coffee_price,price_lag_1
0,2024-04-18,122000,NaN
1,2024-04-19,121000,122000.0
2,2024-04-20,123500,121000.0
3,2024-04-21,123500,123500.0
4,2024-04-22,127000,123500.0


Tương tự, `shift(7)` là giá của 7 ngày trước. 7 dòng đầu sẽ là `NaN`.

In [ ]:
demo["price_lag_7"] = price.shift(7)
demo.head(10)

,date,coffee_price,price_lag_1,price_lag_7
0,2024-04-18,122000,NaN,NaN
1,2024-04-19,121000,122000.0,NaN
2,2024-04-20,123500,121000.0,NaN
3,2024-04-21,123500,123500.0,NaN
4,2024-04-22,127000,123500.0,NaN
5,2024-04-23,129000,127000.0,NaN
6,2024-04-24,129300,129000.0,NaN
7,2024-04-25,132000,129300.0,122000.0
8,2024-04-26,134000,132000.0,121000.0
9,2024-04-27,131500,134000.0,123500.0


Giờ thêm cột lag thật vào bảng `features`. Vòng lặp tạo cả 6 cột lag (1, 2, 3, 7, 14, 30) và làm đúng việc ta vừa thử bằng tay trên `demo`.

In [ ]:
for lag in (1, 2, 3, 7, 14, 30):
    features[f"price_lag_{lag}"] = price.shift(lag)

[c for c in features.columns if c.startswith("price_lag_")]
features.head(15)

,date,coffee_price,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,precipitation_hours,relative_humidity_2m_mean,soil_moisture_0_to_7cm,soil_moisture_7_to_28cm,shortwave_radiation_sum,price_lag_1,price_lag_2,price_lag_3,price_lag_7,price_lag_14,price_lag_30
0,2024-04-18,122000,35.8,24.4,29.3,0.7,0.7,6.0,60,0.247,0.271,22.62,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-04-19,121000,36.4,24.5,29.5,0.5,0.5,2.0,61,0.253,0.271,23.75,122000.0,NaN,NaN,NaN,NaN,NaN
2,2024-04-20,123500,35.8,24.6,29.2,0.8,0.8,4.0,62,0.266,0.271,22.42,121000.0,122000.0,NaN,NaN,NaN,NaN
3,2024-04-21,123500,35.6,25.1,29.6,0.9,0.9,3.0,62,0.275,0.271,22.22,123500.0,121000.0,122000.0,NaN,NaN,NaN
4,2024-04-22,127000,36.2,25.2,30.1,0.0,0.0,0.0,60,0.276,0.271,24.14,123500.0,123500.0,121000.0,NaN,NaN,NaN
5,2024-04-23,129000,36.6,25.3,30.0,1.4,1.4,2.0,62,0.280,0.271,22.51,127000.0,123500.0,123500.0,NaN,NaN,NaN
6,2024-04-24,129300,37.7,25.8,31.2,0.4,0.4,2.0,58,0.280,0.271,24.90,129000.0,127000.0,123500.0,NaN,NaN,NaN
7,2024-04-25,132000,36.5,26.0,30.4,0.5,0.5,1.0,61,0.280,0.271,22.94,129300.0,129000.0,127000.0,122000.0,NaN,NaN
8,2024-04-26,134000,36.5,25.9,30.1,0.2,0.2,1.0,62,0.278,0.271,20.98,132000.0,129300.0,129000.0,121000.0,NaN,NaN
9,2024-04-27,131500,37.5,25.5,31.3,0.0,0.0,0.0,57,0.274,0.271,24.69,134000.0,132000.0,129300.0,123500.0,NaN,NaN


### Feature loại 2: giá trung bình trong N ngày qua

Trước tiên tạo `history` = giá tính đến **hôm qua** (dịch xuống 1 dòng). Dùng hôm qua thay vì hôm nay để model **không nhìn thấy trước** giá của chính ngày cần đoán.

In [ ]:
history = price.shift(1)
history.head(10)

,coffee_price
0,NaN
1,122000.0
2,121000.0
3,123500.0
4,123500.0
5,127000.0
6,129000.0
7,129300.0
8,132000.0
9,134000.0


`history.rolling(7).mean()` tính trung bình của **7 ngày liên tiếp** gần nhất. 7 dòng đầu chưa đủ 7 ngày nên là `NaN`.

In [ ]:
demo = features[["date", "coffee_price"]].copy()
demo["price_mean_7"] = history.rolling(7).mean()
demo.head(10)

,date,coffee_price,price_mean_7
0,2024-04-18,122000,NaN
1,2024-04-19,121000,NaN
2,2024-04-20,123500,NaN
3,2024-04-21,123500,NaN
4,2024-04-22,127000,NaN
5,2024-04-23,129000,NaN
6,2024-04-24,129300,NaN
7,2024-04-25,132000,125042.857143
8,2024-04-26,134000,126471.428571
9,2024-04-27,131500,128328.571429


Bây giờ thêm vào bảng `features` các cột trung bình 7, 14, 30 ngày. Ngoài ra tạo thêm **độ lệch chuẩn** (`std`) cho cửa sổ 7 và 30 ngày: số này càng lớn nghĩa là giá càng **biến động mạnh**.

In [43]:
for window in (7, 14, 30):
    history = price.shift(1)
    features[f"price_mean_{window}"] = history.rolling(window).mean()
    if window in (7, 30):
        features[f"price_std_{window}"] = history.rolling(window).std()

features[["date", "price_mean_7", "price_std_7", "price_mean_14", "price_mean_30", "price_std_30"]].iloc[28:34]

,date,price_mean_7,price_std_7,price_mean_14,price_mean_30,price_std_30
28,2024-05-16,100185.714286,1393.351561,103485.714286,NaN,NaN
29,2024-05-17,100685.714286,1068.376606,101414.285714,NaN,NaN
30,2024-05-18,101171.428571,939.351347,100300.000000,115356.666667,14896.439828
31,2024-05-19,101800.000000,1093.922605,100364.285714,114753.333333,14984.376538
32,2024-05-20,102214.285714,1274.661486,100571.428571,114183.333333,15063.591259
33,2024-05-21,102642.857143,1302.378776,100857.142857,113533.333333,15068.426304


### Feature loại 3: thời tiết trung bình trong 7 và 30 ngày qua

Cùng cách làm như giá: dịch 1 dòng rồi tính trung bình trượt. Thử với lượng mưa trước.

In [44]:
demo = features[["date", "rain_sum"]].copy()
demo["rain_7d"] = features["rain_sum"].shift(1).rolling(7).mean()
demo.head(10)

,date,rain_sum,rain_7d
0,2024-04-18,0.7,NaN
1,2024-04-19,0.5,NaN
2,2024-04-20,0.8,NaN
3,2024-04-21,0.9,NaN
4,2024-04-22,0.0,NaN
5,2024-04-23,1.4,NaN
6,2024-04-24,0.4,NaN
7,2024-04-25,0.5,0.671429
8,2024-04-26,0.2,0.642857
9,2024-04-27,0.0,0.600000


Dùng vòng lặp làm cho cả 3 chỉ số: mưa, nhiệt độ, độ ẩm. Mỗi chỉ số tạo ra 2 cột (`_7d` và `_30d`).

In [45]:
for column, name in (
    ("rain_sum", "rain"),
    ("temperature_2m_mean", "temperature_mean"),
    ("relative_humidity_2m_mean", "humidity_mean"),
):
    features[f"{name}_7d"] = features[column].shift(1).rolling(7).mean()
    features[f"{name}_30d"] = features[column].shift(1).rolling(30).mean()

[c for c in features.columns if c.endswith("_7d") or c.endswith("_30d")]

['rain_7d',
 'rain_30d',
 'temperature_mean_7d',
 'temperature_mean_30d',
 'humidity_mean_7d',
 'humidity_mean_30d']

### Feature loại 4: thông tin theo lịch (mùa vụ)

`.dt` giúp lấy từng phần của ngày: thứ trong tuần (0 = thứ Hai), ngày trong tháng, tháng, quý.

In [46]:
features["day_of_week"] = features["date"].dt.dayofweek
features["day_of_month"] = features["date"].dt.day
features["month"] = features["date"].dt.month
features["quarter"] = features["date"].dt.quarter
features[["date", "day_of_week", "day_of_month", "month", "quarter"]].head()

,date,day_of_week,day_of_month,month,quarter
0,2024-04-18,3,18,4,2
1,2024-04-19,4,19,4,2
2,2024-04-20,5,20,4,2
3,2024-04-21,6,21,4,2
4,2024-04-22,0,22,4,2


### Target: đáp án mà model phải đoán

`shift(-1)` dịch **lên** 1 dòng (ngược với `shift(1)`), nên mỗi dòng sẽ thấy giá của **ngày kế tiếp**. Dòng cuối cùng không có ngày kế tiếp nên là `NaN`.

In [48]:
features["next_day_price"] = price.shift(-1)
features[["date", "coffee_price", "next_day_price"]].tail(5)

,date,coffee_price,next_day_price
889,2026-09-24,92800,92500.0
890,2026-09-25,92500,93600.0
891,2026-09-26,93600,93600.0
892,2026-09-27,93600,93600.0
893,2026-09-28,93600,NaN


# Bước cuối: bỏ các dòng còn ô trống

**Bước 4.2** Đếm số ô trống ở mỗi cột (chỉ hiện cột có ô trống). `shift` và `rolling` làm các dòng đầu bị trống, `shift(-1)` làm dòng cuối bị trống.

In [49]:
nan_count = features.isna().sum()
nan_count[nan_count > 0]

,0
price_lag_1,1
price_lag_2,2
price_lag_3,3
price_lag_7,7
price_lag_14,14
price_lag_30,30
price_mean_7,7
price_std_7,7
price_mean_14,14
price_mean_30,30


**Bước 4.3** Bỏ mọi dòng còn ô trống, rồi đánh lại số thứ tự từ 0. So sánh số dòng trước và sau.

In [50]:
rows_before = len(features)
features = features.dropna().reset_index(drop=True)
print("Trước:", rows_before, "| Sau:", len(features))

Trước: 894 | Sau: 863


**Bước 4.4** Bảng cuối cùng: đây chính là `training_dataset` dùng để dạy model.

In [51]:
features.head()

,date,coffee_price,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,precipitation_hours,relative_humidity_2m_mean,soil_moisture_0_to_7cm,...,rain_30d,temperature_mean_7d,temperature_mean_30d,humidity_mean_7d,humidity_mean_30d,day_of_week,day_of_month,month,quarter,next_day_price
0,2024-05-18,103900,31.8,24.0,26.6,16.3,16.3,7.0,84,0.459,...,1.666667,29.071429,29.476667,67.142857,63.800000,5,18,5,2,103900.0
1,2024-05-19,103900,29.0,24.4,26.1,3.0,3.0,7.0,86,0.473,...,2.186667,28.600000,29.386667,70.571429,64.600000,6,19,5,2,104000.0
2,2024-05-20,104000,31.5,24.2,27.0,3.9,3.9,4.0,79,0.461,...,2.270000,28.014286,29.273333,74.142857,65.433333,0,20,5,2,105600.0
3,2024-05-21,105600,27.3,24.3,25.7,1.7,1.7,9.0,85,0.445,...,2.373333,27.542857,29.200000,77.000000,66.000000,1,21,5,2,110000.0
4,2024-05-22,110000,32.3,24.0,27.0,7.4,7.4,9.0,80,0.417,...,2.400000,27.057143,29.070000,79.714286,66.766667,2,22,5,2,117000.0


In [52]:
print(features.shape)
list(features.columns)

(863, 34)


['date',
 'coffee_price',
 'temperature_2m_max',
 'temperature_2m_min',
 'temperature_2m_mean',
 'precipitation_sum',
 'rain_sum',
 'precipitation_hours',
 'relative_humidity_2m_mean',
 'soil_moisture_0_to_7cm',
 'soil_moisture_7_to_28cm',
 'shortwave_radiation_sum',
 'price_lag_1',
 'price_lag_2',
 'price_lag_3',
 'price_lag_7',
 'price_lag_14',
 'price_lag_30',
 'price_mean_7',
 'price_std_7',
 'price_mean_14',
 'price_mean_30',
 'price_std_30',
 'rain_7d',
 'rain_30d',
 'temperature_mean_7d',
 'temperature_mean_30d',
 'humidity_mean_7d',
 'humidity_mean_30d',
 'day_of_week',
 'day_of_month',
 'month',
 'quarter',
 'next_day_price']

## Thư mục lưu trữ theo cấu trúc project

In [55]:
from pathlib import Path

# Dùng lại PROJECT_ROOT đã xác định ở đầu notebook (hoặc fallback an toàn)
if 'PROJECT_ROOT' not in globals():
    current_dir = Path.cwd()
    PROJECT_ROOT = current_dir.parent if (current_dir.parent / "data").exists() else current_dir

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

def _save_csv(df, path):
    df.to_csv(path, index=False, date_format="%Y-%m-%d")
    print(f"{path.stem}: {len(df)} dòng -> {path}")

# Lưu 4 file chuẩn format
_save_csv(coffee, PROCESSED_DIR / "coffee_clean.csv")
_save_csv(weather, PROCESSED_DIR / "weather_clean.csv")
_save_csv(merged, PROCESSED_DIR / "merged_daily.csv")
_save_csv(features, FEATURES_DIR / "training_dataset.csv")

coffee_clean: 894 dòng -> /content/data/processed/coffee_clean.csv
weather_clean: 894 dòng -> /content/data/processed/weather_clean.csv
merged_daily: 894 dòng -> /content/data/processed/merged_daily.csv
training_dataset: 863 dòng -> /content/data/features/training_dataset.csv


## Tổng kết

| Bảng | Là gì |
|---|---|
| `coffee` | Giá cà phê đã sạch (2 cột) |
| `weather` | Thời tiết đã sạch (11 cột) |
| `merged` | Giá + thời tiết ghép theo ngày |
| `features` | `merged` + các cột feature + cột target, đã bỏ dòng trống |

Thử thách: đổi cửa sổ trung bình từ 7 sang 3 ở bước tính `price_mean_7`, chạy lại và xem kết quả khác thế nào.